In [4]:
import pandas as pd
import os

DATA_DIR = r"C:\Users\semwi\FPL-Core-Insights\data"
PLAYER_PATH = os.path.join(DATA_DIR, "player_data.csv")
OUTPUT_PATH = os.path.join(DATA_DIR, "player_strength.csv")

df = pd.read_csv(PLAYER_PATH)

# Alleen rijen waar gespeeld is (minutes > 0)
df = df[df["minutes_played"] > 0].copy()

stat_cols = [
    "touches", "total_pass", "accurate_pass",
    "total_long_balls", "accurate_long_balls", "total_cross", "accurate_cross",
    "key_pass", "total_shots", "on_target", "shot_off_target", "blocked_shot",
    "goals", "goal_assist", "big_chance_created", "big_chance_missed", "hit_woodwork",
    "duel_won", "duel_lost", "aerial_won", "aerial_lost", "total_tackle", "won_tackle",
    "interception_won", "total_clearance", "outfielder_block", "total_contest",
    "won_contest", "dispossessed", "possession_lost", "unsuccessful_touch",
    "ball_recovery", "was_fouled", "fouls", "total_offside", "penalty_won",
    "penalty_conceded", "penalty_miss", "error_led_to_goal", "saves",
    "saves_inside_box", "penalty_save", "punches", "acc_own_half_pass", "acc_opp_half_pass"
]

# Schaal alle stats naar per 90 minuten
for col in stat_cols:
    df[col] = df[col] / df["minutes_played"] * 90

# Info per speler
info = df.groupby("player_id").agg(
    player_name   = ("player_name",    "first"),
    short_name    = ("short_name",     "first"),
    nationality   = ("nationality",    "first"),
    height        = ("height",         "first"),
    position      = ("position",       lambda x: x.mode()[0] if len(x) > 0 else None),
    market_value  = ("market_value",   "last"),
    matches_played= ("match_id",       "nunique"),
    avg_minutes   = ("minutes_played", "mean"),
).reset_index()

# Gemiddelde per 90 min stats
avgs = df.groupby("player_id")[stat_cols].mean().round(2).reset_index()

# Rating apart — gewoon gemiddelde van wedstrijden met >0 min (al gefilterd)
ratings = df.groupby("player_id")["rating"].mean().round(2).reset_index()

# Combineer
result = info.merge(avgs, on="player_id").merge(ratings, on="player_id")
result["avg_minutes"] = result["avg_minutes"].round(1)
result = result.sort_values("matches_played", ascending=False).reset_index(drop=True)

result.to_csv(OUTPUT_PATH, index=False)
print(f"✅ player_strength.csv: {len(result)} spelers opgeslagen (per 90 min)")

C:\Users\semwi\AppData\Local\Temp\ipykernel_8000\2116252321.py:8: DtypeWarning: Columns (15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(PLAYER_PATH)


✅ player_strength.csv: 1478 spelers opgeslagen (per 90 min)


In [5]:
import pandas as pd
import os

DATA_DIR      = r"C:\Users\semwi\FPL-Core-Insights\data"
PLAYER_PATH   = os.path.join(DATA_DIR, "player_data.csv")
STRENGTH_PATH = os.path.join(DATA_DIR, "player_strength.csv")
OUTPUT_PATH   = os.path.join(DATA_DIR, "team_strength.csv")

players  = pd.read_csv(PLAYER_PATH)
strength = pd.read_csv(STRENGTH_PATH)[["player_id", "rating"]]

# Alleen basisspelers (geen invaller)
starters = players[players["substitute"] == False].copy()

# Koppel player strength rating
starters = starters.merge(strength, on="player_id", how="left", suffixes=("_match", "_avg"))

# Gemiddelde strength rating per team per wedstrijd
team_strength = (
    starters.groupby(["match_id", "season", "round", "timestamp", "home_team", "away_team", "side"])
    ["rating_avg"].mean().round(3).reset_index()
    .rename(columns={"rating_avg": "team_strength", "side": "team_side"})
)

# Voeg teamnaam toe als losse kolom
team_strength["team"] = team_strength.apply(
    lambda r: r["home_team"] if r["team_side"] == "home" else r["away_team"], axis=1
)

team_strength = team_strength.sort_values(["timestamp", "match_id"]).reset_index(drop=True)

team_strength.to_csv(OUTPUT_PATH, index=False)
print(f"✅ team_strength.csv: {len(team_strength)} rijen opgeslagen")
print(team_strength.head(10).to_string(index=False))

C:\Users\semwi\AppData\Local\Temp\ipykernel_8000\3778048941.py:9: DtypeWarning: Columns (15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  players  = pd.read_csv(PLAYER_PATH)


✅ team_strength.csv: 5013 rijen opgeslagen
 match_id    season  round        timestamp       home_team              away_team team_side  team_strength                   team
  8243388 2019-2020      1 2019-08-09 19:00       Liverpool           Norwich City      away          6.703           Norwich City
  8243388 2019-2020      1 2019-08-09 19:00       Liverpool           Norwich City      home          7.035              Liverpool
  8243390 2019-2020      1 2019-08-10 11:30 West Ham United        Manchester City      away          7.157        Manchester City
  8243390 2019-2020      1 2019-08-10 11:30 West Ham United        Manchester City      home          6.807        West Ham United
  8243391 2019-2020      1 2019-08-10 14:00  Crystal Palace                Everton      away          6.843                Everton
  8243391 2019-2020      1 2019-08-10 14:00  Crystal Palace                Everton      home          6.795         Crystal Palace
  8243395 2019-2020      1 2019-08-10 14